***Objectives:***

Create an AI agent that can process user messages and utilize tools effectively
Implement a calculator tool and a data analysis tool within the agent
Test the agent's functionality with various scenarios to ensure it operates as expected

In [14]:
!pip install langchain-openai
from typing import List, Any
from dotenv import load_dotenv
import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

In [ ]:
class Agent:
    """An AI Agent that can use tools to help answer questions"""

    def __init__(
        self,
        role: str = "Personal Assistant",
        instructions: str = "Help users with any question",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[Any] = None
    ):

        self.model = model
        self.role = role
        self.instructions = instructions
        self.tools = tools or []

        self.llm = ChatOpenAI(
            model=model,
            api_key="",
            temperature=temperature
        )

    def invoke(self, user_message: str) -> str:
        """Process a user message and return a response"""

        messages = [
            SystemMessage(
                content=f"You're an AI Agent and your role is {self.role}. "
                        f"Your instructions: {self.instructions}"
            )
        ]

        messages.append(HumanMessage(content=user_message))

        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        while hasattr(ai_message, "tool_calls") and ai_message.tool_calls:

            for call in ai_message.tool_calls:

                function_name = call["name"]
                function_args = call["args"]
                tool_call_id = call["id"]

                tool = next(
                    (t for t in self.tools if t.name == function_name),
                    None
                )

                if tool:
                    result = tool(**function_args)

                    messages.append(
                        ToolMessage(
                            content=json.dumps(result),
                            tool_call_id=tool_call_id
                        )
                    )

            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)

        return ai_message.content

In [17]:
agent = Agent(role="Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.


In [19]:
def calculate(expression: str) -> float:
    """Evaluate a mathematical expression"""
    return eval(expression)

math_agent = Agent(role="Math Assistant", tools=[calculate])
response = math_agent.invoke("What is 23 * 45?")
print(response)

23 * 45 = 1,035.


In [20]:

def get_games(num_games:int=1, top:bool=True) -> str:
    """Returns the top or bottom N games with highest or lowest scores."""
    # Game data and sorting logic
    ...

data_analyst_agent = Agent(role="Game Stats Assistant", instructions="You can bring insights about a game dataset based on users questions", tools=[get_games])
response = data_analyst_agent.invoke("What's the best game in the dataset?")
print(response)

To determine the "best" game in the dataset, we would typically look at various metrics such as user ratings, sales figures, critical reviews, and player engagement. However, without specific criteria for what "best" means (e.g., highest rating, most sales, etc.), I can't provide a definitive answer.

If you have specific metrics or criteria in mind, please let me know, and I can help identify the best game based on that information!
